<a href="https://colab.research.google.com/github/tzmudder/AAI2026/blob/dev/Prompt_Engineering_Part_2_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import re
import textwrap
from datetime import datetime

# ---------------------------
# 1) Sample data (with edge cases)
# ---------------------------
data = [
    {
        "request_id": "SR-1001",
        "created_at": "2026-03-01 09:12:00",
        "customer": "Acme Co",
        "channel": "email",
        "message": "Hi team, our invoice export is failing for one user. Not urgent, but we'd like a fix this week."
    },
    {
        "request_id": "SR-1002",
        "created_at": "2026-03-01 09:18:00",
        "customer": "BluePeak",
        "channel": "chat",
        "message": "This is ridiculous. We can’t log in and it’s blocking the entire team from working. Fix it now."
    },
    {
        "request_id": "SR-1003",
        "created_at": "BAD_DATE",  # bad date
        "customer": "Northwind",
        "channel": "phone",
        "message": "We think there may have been unauthorized access. We see suspicious password reset emails. Please respond ASAP."
    },
    {
        "request_id": "SR-1004",
        "created_at": None,  # missing date
        "customer": "Contoso",
        "channel": "web",
        "message": "Feature request: can you add dark mode? No rush."
    },
    {
        "request_id": "SR-1005",
        "created_at": "2026-03-01 09:41:00",
        "customer": "Globex",
        "channel": "email",
        "message": None  # missing message
    },
]

df = pd.DataFrame(data)

# ---------------------------
# 2) Cleaning: dates + missing values
# ---------------------------
df["created_at_parsed"] = pd.to_datetime(df["created_at"], errors="coerce")
df["message_clean"] = df["message"].fillna("").astype(str)

# ---------------------------
# 3) Rule-based tone + impact classification
# ---------------------------
TONE_KEYWORDS = {
    "panicked": [
        "asap", "immediately", "urgent", "right now", "now", "security", "breach",
        "unauthorized", "suspicious", "hacked", "compromised"
    ],
    "angry": [
        "ridiculous", "unacceptable", "fix it now", "terrible", "angry", "furious",
        "this is a joke", "outrageous"
    ],
    "frustrated": [
        "can't", "cannot", "won't", "broken", "blocking", "stuck", "issue", "problem",
        "frustrated", "fail", "failing", "error"
    ],
}

IMPACT_KEYWORDS = {
    "critical": [
        "entire team", "all users", "system down", "outage", "production down",
        "breach", "unauthorized access", "data loss"
    ],
    "high": [
        "blocking", "can't log in", "cannot log in", "cannot login", "can't login",
        "payments", "orders", "revenue"
    ],
    "medium": [
        "one user", "some users", "degraded", "slow", "fails", "failing"
    ],
    "low": [
        "feature request", "no rush", "nice to have", "this week", "when you can"
    ],
}

SECURITY_KEYWORDS = ["unauthorized", "breach", "suspicious", "security", "hacked", "compromised", "password reset"]

def _contains_any(text: str, keywords: list[str]) -> bool:
    return any(k in text for k in keywords)

def classify_tone(message: str) -> tuple[str, list[str]]:
    msg = message.lower()
    # Priority order: panicked > angry > frustrated > calm
    for tone in ["panicked", "angry", "frustrated"]:
        hits = [k for k in TONE_KEYWORDS[tone] if k in msg]
        if hits:
            return tone, hits[:5]
    return "calm", []

def classify_impact(message: str) -> tuple[str, list[str]]:
    msg = message.lower()
    # Priority order: critical > high > medium > low
    for impact in ["critical", "high", "medium", "low"]:
        hits = [k for k in IMPACT_KEYWORDS[impact] if k in msg]
        if hits:
            return impact, hits[:5]
    return "low", []

def classify_security_flag(message: str) -> bool:
    msg = message.lower()
    return _contains_any(msg, SECURITY_KEYWORDS)

# Apply classifiers
tone_results = df["message_clean"].apply(classify_tone)
impact_results = df["message_clean"].apply(classify_impact)

df["tone"] = tone_results.apply(lambda x: x[0])
df["tone_evidence"] = tone_results.apply(lambda x: x[1])

df["operational_impact"] = impact_results.apply(lambda x: x[0])
df["impact_evidence"] = impact_results.apply(lambda x: x[1])

df["security_flag"] = df["message_clean"].apply(classify_security_flag)

# ---------------------------
# 4) Scoring + urgency mapping
# ---------------------------
TONE_SCORE = {"calm": 0, "frustrated": 1, "angry": 2, "panicked": 3}
IMPACT_SCORE = {"low": 0, "medium": 2, "high": 4, "critical": 6}

def decide_urgency(tone: str, impact: str, security_flag: bool) -> tuple[str, str, int]:
    score = TONE_SCORE.get(tone, 0) + IMPACT_SCORE.get(impact, 0)
    if security_flag:
        score += 3

    if score >= 9:
        return "P0 - Critical (Immediate Response)", "Respond ≤ 15 min", score
    elif score >= 6:
        return "P1 - High (Urgent)", "Respond ≤ 1 hour", score
    elif score >= 3:
        return "P2 - Medium (Standard)", "Respond ≤ 1 business day", score
    else:
        return "P3 - Low (Backlog/Planned)", "Respond ≤ 3 business days", score

urg = df.apply(lambda r: decide_urgency(r["tone"], r["operational_impact"], r["security_flag"]), axis=1)
df["urgency_level"] = urg.apply(lambda x: x[0])
df["sla_target"] = urg.apply(lambda x: x[1])
df["score"] = urg.apply(lambda x: x[2])

# ---------------------------
# 5) Write a clean text report
# ---------------------------
def build_report(df: pd.DataFrame) -> str:
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    lines = []
    lines.append("SERVICE DESK TRIAGE REPORT")
    lines.append(f"Generated: {now}")
    lines.append("=" * 72)

    # Sort by score (highest urgency first), then date (newest first)
    df_sorted = df.sort_values(by=["score", "created_at_parsed"], ascending=[False, False], na_position="last")

    for _, r in df_sorted.iterrows():
        created = r["created_at_parsed"]
        created_str = created.strftime("%Y-%m-%d %H:%M:%S") if pd.notna(created) else "UNKNOWN_DATE"

        lines.append(f"Request: {r['request_id']} | Customer: {r['customer']} | Channel: {r['channel']} | Created: {created_str}")
        lines.append(f"Urgency: {r['urgency_level']}  | SLA: {r['sla_target']}  | Score: {int(r['score'])}")
        lines.append(f"Tone: {r['tone']} | Impact: {r['operational_impact']} | Security Flag: {bool(r['security_flag'])}")

        evidence_parts = []
        if isinstance(r["tone_evidence"], list) and r["tone_evidence"]:
            evidence_parts.append("tone=" + ", ".join(r["tone_evidence"]))
        if isinstance(r["impact_evidence"], list) and r["impact_evidence"]:
            evidence_parts.append("impact=" + ", ".join(r["impact_evidence"]))
        if evidence_parts:
            lines.append("Evidence: " + " | ".join(evidence_parts))

        msg = r["message_clean"].strip() or "[NO MESSAGE PROVIDED]"
        lines.append("Message:")
        lines.append(textwrap.fill(msg, width=72))
        lines.append("-" * 72)

    return "\n".join(lines)

report_text = build_report(df)

out_path = "service_triage_report.txt"
with open(out_path, "w", encoding="utf-8") as f:
    f.write(report_text)

# Preview results
print(df[["request_id", "created_at", "tone", "operational_impact", "security_flag", "urgency_level", "sla_target", "score"]])
print("\n--- REPORT PREVIEW (first ~60 lines) ---\n")
print("\n".join(report_text.splitlines()[:60]))
print(f"\nSaved report to: {out_path}")

  request_id           created_at      tone operational_impact  security_flag  \
0    SR-1001  2026-03-01 09:12:00  panicked             medium          False   
1    SR-1002  2026-03-01 09:18:00  panicked           critical          False   
2    SR-1003             BAD_DATE  panicked           critical           True   
3    SR-1004                 None      calm                low          False   
4    SR-1005  2026-03-01 09:41:00      calm                low          False   

                        urgency_level                 sla_target  score  
0              P2 - Medium (Standard)   Respond ≤ 1 business day      5  
1  P0 - Critical (Immediate Response)           Respond ≤ 15 min      9  
2  P0 - Critical (Immediate Response)           Respond ≤ 15 min     12  
3          P3 - Low (Backlog/Planned)  Respond ≤ 3 business days      0  
4          P3 - Low (Backlog/Planned)  Respond ≤ 3 business days      0  

--- REPORT PREVIEW (first ~60 lines) ---

SERVICE DESK TRIAGE REPORT